In [ ]:
import sys

sys.path.append('..')
sys.path.append("../Assessor models/Error-risk assessor")

In [ ]:
from pathlib import Path

DATA_PATH = Path("../data")
RAW_DATA_PATH = DATA_PATH / "raw"

RESULTS_PATH = RAW_DATA_PATH / "results"
CONFIDENCE_PATH = RAW_DATA_PATH / "confidence" # path to Data.zip ...

ASER_RESULTS_PATH = RESULTS_PATH / "assessor-error_risk"
ASAR_RESULTS_PATH = RESULTS_PATH / "assessor-answerability_risk"
BB_RESULTS_PATH = RESULTS_PATH / "black-box"
GB_RESULTS_PATH = RESULTS_PATH / "grey-box"

for path in (
    ASER_RESULTS_PATH,
    ASAR_RESULTS_PATH,
    BB_RESULTS_PATH,
    GB_RESULTS_PATH,
):
    path.mkdir(parents=True, exist_ok=True)
    (path / "IID").mkdir(parents=True, exist_ok=True)
    (path / "OOD").mkdir(parents=True, exist_ok=True)

# Results

In [ ]:
from itertools import product
import pandas as pd
import numpy as np
from modeling.utils import LabelEncoderWithReject
from modeling.evaluation import RejectionAnalyzer


def yield_combinations(**spaces):
    keys = list(spaces.keys())
    values = list(spaces.values())
    for combination in product(*values):
        yield dict(zip(keys, combination))


def read_confidence_csv(distribution, engine, benchmark, feature=None, variant=None, assessor=True):
    path = CONFIDENCE_PATH / distribution / engine / benchmark 
    fname = ""
    fname += "asr_" if assessor else ""
    fname += f"{feature}_" if feature else ""
    fname += f"{variant}_" if variant else ""
    fname = fname.rstrip('_') + ".csv"
    path = path / fname
    if not path.exists():
        return None
    df = pd.read_csv(path).dropna(subset=['confidence', 'outcome'])
    return df


def build_probas(confidence, le):
    risk_score = 1 - (confidence / 100.0)
    probs = np.zeros((len(risk_score), 3))
    probs[:, le.transform(['INCORRECT'])[0]] = risk_score
    return probs


def evaluate_method(y_true, probs, le):
    ra = RejectionAnalyzer(label_encoder=le)
    analyzer = ra(y_true=y_true, probs=probs)
    return analyzer.evaluate()


def metrics_to_df(metrics_dict):
    data = {'partition': [], 'metric': [], 'value': [], 'margin': []}
    for metric, value in metrics_dict.items():
        if metric.endswith('_margin'):
            continue
        data['partition'].append('val')
        data['metric'].append(metric)
        data['value'].append(value)
        data['margin'].append(metrics_dict.get(metric + '_margin', None))
    return pd.DataFrame(data)


def add_constants_to_df(df, **constants):
    for key, value in constants.items():
        df[key] = value
    return df

In [96]:
from tqdm import tqdm

BENCHMARKS = [
    'addition',
    'anagram',
    'locality',
    'science',
    'logic_puzzles',
    'geometry'
]

ENGINES = [
    'gemini-3-pro-preview',
    'GPT5.2',
    'llama3v3chat-70B',
    'Qwen2.5_72B'
]

FEATURES = [
    'difficulty',
    'embedding',
    'demands',
    'difficulty+embedding',
    'difficulty+demands',
    'embedding+demands',
    'difficulty+embedding+demands'
]

ASER2ASAR_FEATURE_MAPPING ={
    'difficulty': 'diff',
    'embedding': 'emb',
    'demands': 'adele',
    'difficulty+embedding': 'emb+diff',
    'difficulty+demands': 'diff+adele',
    'embedding+demands': 'emb+adele',
    'difficulty+embedding+demands': 'emb+diff+adele'
}

ASAR2ASER_FEATURE_MAPPING = {
    v: k for k, v in ASER2ASAR_FEATURE_MAPPING.items()
}

BB_VARIANTS = [
    'vanilla',
    'CoT',
    'self_probing',
    'multi_step',
    'top_k',
    'self_consistency'
]

GB_VARIANTS = [
    'SAR',
    'semantic_energy'
]

METHOD_SPACES = {
    'ASAR': {
        'engine': ENGINES,
        'benchmark': BENCHMARKS,
        'variant': BB_VARIANTS+GB_VARIANTS,
        'feature': [ASER2ASAR_FEATURE_MAPPING[f] for f in FEATURES]
    },
    'ASER': {
        'engine': ENGINES,
        'benchmark': BENCHMARKS,
        'feature': FEATURES},
    'BB': {
        'engine': ENGINES,
        'benchmark': BENCHMARKS,
        'variant': BB_VARIANTS
    },
    'GB': {
        'engine': ENGINES,
        'benchmark': BENCHMARKS,
        'variant': GB_VARIANTS
    }
}

def build_results(method, distribution):
    le = LabelEncoderWithReject()
    le.fit(['CORRECT', 'INCORRECT', 'AVOIDANCE'])
    dfs = []
    for combination in tqdm(yield_combinations(
        **METHOD_SPACES[method]),
        desc=f"{method} - {distribution}"
    ):
        df = read_confidence_csv(
            distribution,
            **combination,
            assessor=(method in ['ASAR', 'ASER'])
        )
        if df is None:
            continue
        probs = build_probas(df['confidence'].values, le)
        y_true = le.transform(df['outcome'])
        metrics = evaluate_method(y_true, probs, le)
        mdf = metrics_to_df(metrics)
        mdf = add_constants_to_df(
            mdf,
            engine=combination['engine'],
            benchmark=combination['benchmark'],
            variant=combination.get('variant', None),
            features=ASAR2ASER_FEATURE_MAPPING.get(
                combination.get('feature', None),
                combination.get('feature', None)
            ), 
            num_instances=len(df)
        )
        dfs.append(mdf)
    results = pd.concat(dfs, ignore_index=True)
    results.loc[(results['metric'] == 'FOC-AUC') & (results['value'] == 0), 'value'] = np.nan
    tgt_cols = ['engine', 'benchmark', 'variant', 'features', 'metric', 'value', 'margin', 'num_instances']
    return results[tgt_cols]

def save_results(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

In [98]:
METHODS = [
    ('ASER', ASER_RESULTS_PATH),
    ('ASAR', ASAR_RESULTS_PATH),
    ('BB', BB_RESULTS_PATH),
    ('GB', GB_RESULTS_PATH)
]
DISTRIBUTIONS = ['IID', 'OOD']

for method, results_path in METHODS:
    for distribution in DISTRIBUTIONS:
        if distribution == 'OOD' and method in ['BB', 'GB']:
            continue
        res = build_results(method, distribution)
        if method in ['ASAR', 'ASER']:
            save_results(res, results_path / distribution / "metrics.csv")
        else:
            save_results(res, results_path / "metrics.csv")

ASER - IID: 168it [00:07, 23.57it/s]
ASER - OOD: 168it [00:07, 21.36it/s]
ASAR - IID: 1344it [01:24, 15.89it/s]
ASAR - OOD: 1344it [01:26, 15.50it/s]
BB - IID: 144it [00:13, 10.95it/s]
GB - IID: 48it [00:00, 51.26it/s] 


# Metrics

In [117]:
def get_metrics(distribution):
    dfs = []
    for method, method_path in METHODS:
        if method in ['ASER', 'ASAR']:
            method_path /=  distribution 
        df = pd.read_csv(method_path / "metrics.csv")
        df.insert(0, 'method', method)
        dfs.append(df)
    metrics_df = pd.concat(dfs, ignore_index=True)
    metrics_df['method'] = pd.Categorical(metrics_df['method'], categories=[m[0] for m in METHODS], ordered=True)
    metrics_df['engine'] = pd.Categorical(metrics_df['engine'], categories=ENGINES, ordered=True)
    metrics_df['benchmark'] = pd.Categorical(metrics_df['benchmark'], categories=BENCHMARKS, ordered=True)
    metrics_df['features'] = pd.Categorical(metrics_df['features'].fillna('-'), categories=FEATURES+['-'], ordered=True)
    metrics_df['variant'] = pd.Categorical(metrics_df['variant'].fillna('-'), categories=BB_VARIANTS+GB_VARIANTS+['-'], ordered=True)
    return metrics_df

metrics_iid = get_metrics('IID')
metrics_ood = get_metrics('OOD')
print(f"IID metrics: {metrics_iid.shape[0]} rows")
print(f"OOD metrics: {metrics_ood.shape[0]} rows")

IID metrics: 55178 rows
OOD metrics: 55178 rows


In [ ]:
METHOD_FORMATTER = {
    'ASER': r"$\mathcal{A_{ER}}$",
    'ASAR': r"$\mathcal{A_{AR}}$",
    'BB': r"Black-Box",
    'GB': r"Grey-Box"
}

VARIANT_FORMATTER = {
    'vanilla': r"\textit{Vanilla}",
    'CoT': r"\textit{CoT}",
    'self_probing': r"\textit{Self-Prob}",
    'multi_step': r"\textit{Multi-Step}",
    'top_k': r"\textit{Top-K}",
    'self_consistency': r"\textit{Self-Cons}",
    'SAR': r"\textit{SAR}",
    'semantic_energy': r"\textit{Sema-Energy}"
}

METRIC_FORMATTER = {
        'original-error': r'\textbf{Orig. Error}$\downarrow$',
        "error@MAX-ERG": r"\textbf{Err}$\downarrow$",
        "PVR@MAX-ERG": r"\textbf{Cov}$\uparrow$",
        "SAR@MAX-ERG": r"\textbf{SAR}$\uparrow$",
        "ERR@MAX-ERG": r"\textbf{URR}$\uparrow$",
}

def format_feats(feat_str):
    if pd.isna(feat_str):
        return '-'
    feat_str = feat_str.replace("difficulty", r"\textit{Diff}")
    feat_str = feat_str.replace("embedding", r"\textit{Emb}")
    feat_str = feat_str.replace("demands", r"\textit{DeLeAn}")
    return feat_str

def format_engine(engine_str):
    return f"\modelt{engine_str}"

def bold_max_per_method_and_benchmark(df, subset=None, direction={}, method_col='method'):
    df_out = df.copy()
    for method in df.index.get_level_values(method_col).unique():
        method_slice = df.xs(method, level=method_col)
        for col in df.columns:
            if subset is not None and col not in subset:
                continue
            if direction.get(col, 'max') == 'max':
                val = method_slice[col].max()
            else:
                val = method_slice[col].min()
            mask = (
                (df.index.get_level_values(method_col) == method) &
                (df[col] == val)
            )
            df_out.loc[mask, col] = (
                df_out.loc[mask, col]
                .astype(float)
                .map(lambda x: f"\\textbf{{{x:.3f}}}")
            )
    for col in df_out.columns:
        df_out[col] = df_out[col].apply(
            lambda x: f"{x:.3f}" if isinstance(x, float) else x
        )
    return df_out

#### Table 1: Summary table @ ORP

In [128]:
import pandas as pd

def format_summary_table(summary_df, distribution='IID'):
    formatted = summary_df.copy()
    def format_method_id(method_id):
        method, variant, features = method_id.split('|')
        method = METHOD_FORMATTER.get(method.strip(), method.strip())
        variant = VARIANT_FORMATTER.get(variant.strip(), variant.strip())
        features = format_feats(features.strip())
        return f"{method} | {variant} | {features}"
    formatted['method_id'] = formatted['method_id'].apply(format_method_id)
    formatted['engine'] = formatted['engine'].apply(format_engine)
    formatted.columns = [
        METRIC_FORMATTER.get(col, col) if col not in ['model', 'method'] else col
        for col in formatted.columns
        
    ]
    caption = f"Summary table of best methods per model based on median %s ERG@MAX-ERG across benchmarks."
    if distribution == 'IID':
        caption = caption % "in distribution"
    else:
        caption = caption % "out of distribution"
    formatted = formatted.to_latex(
        float_format="%.3f",
        na_rep="nan",
        caption=caption,
        label="tab:summary_table_best_methods_per_model",
        bold_rows=True,
        escape=False,
        index=False
    )
    return formatted

def summary_table(df, distribution='IID'):
    target_metrics = [
        "original-error",
        "ACC@MAX-ERG",
        "PVR@MAX-ERG",
        "SAR@MAX-ERG",
        "ERR@MAX-ERG",
        "ERG@MAX-ERG",
        "FOC-AUC"
    ]

    df_filt = df[df["metric"].isin(target_metrics)].copy()

    df_filt["method_id"] = (
        df_filt["method"].astype(str) + " | " +
        df_filt["variant"].astype(str) + " | " +
        df_filt["features"].astype(str)
    )

    pivot = (
        df_filt
        .pivot_table(
            index=["engine", "benchmark", "method_id"],
            columns="metric",
            values="value",
            aggfunc="median"
        )
        .reset_index()
    )

    mean_by_method = (
        pivot
        .groupby(["engine", "method_id"])
        .median(numeric_only=True)
        .reset_index()
    )

    best_methods = (
        mean_by_method
        .sort_values("ERG@MAX-ERG", ascending=False)
        .groupby("engine", as_index=False)
        .first()
        .sort_values("engine")
    )

    best_methods['error@MAX-ERG'] = 1 - best_methods['ACC@MAX-ERG']
    summary = best_methods[
        [
            "engine", "method_id", 
            "original-error", "error@MAX-ERG", 
            "PVR@MAX-ERG", "SAR@MAX-ERG", "ERR@MAX-ERG"
        ]
    ]
    return summary, format_summary_table(summary, distribution)

In [129]:
summary_iid, summary_iid_formatted = summary_table(metrics_iid, distribution='IID')
summary_iid.round(3)

/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/3628073018.py:66: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["engine", "method_id"])
/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/3628073018.py:74: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("engine", as_index=False)


metric,engine,method_id,original-error,error@MAX-ERG,PVR@MAX-ERG,SAR@MAX-ERG,ERR@MAX-ERG
0,gemini-3-pro-preview,ASAR | multi_step | difficulty+embedding,0.122,0.018,0.505,0.561,0.926
1,GPT5.2,ASER | - | difficulty+embedding,0.468,0.132,0.230,0.427,0.883
2,llama3v3chat-70B,ASER | - | difficulty+embedding,0.646,0.257,0.190,0.399,0.927
3,Qwen2.5_72B,ASER | - | difficulty+embedding,0.546,0.246,0.210,0.416,0.903


In [130]:
print(summary_iid_formatted)

\begin{table}
\caption{Summary table of best methods per model based on median in distribution ERG@MAX-ERG across benchmarks.}
\label{tab:summary_table_best_methods_per_model}
\begin{tabular}{llrrrrr}
\toprule
engine & method_id & \textbf{Orig. Error}$\downarrow$ & \textbf{Err}$\downarrow$ & \textbf{Cov}$\uparrow$ & \textbf{SAR}$\uparrow$ & \textbf{URR}$\uparrow$ \\
\midrule
\modeltgemini-3-pro-preview & $\mathcal{A_{AR}}$ | \textit{Multi-Step} | \textit{Diff}+\textit{Emb} & 0.122 & 0.018 & 0.505 & 0.561 & 0.926 \\
\modeltGPT5.2 & $\mathcal{A_{ER}}$ | - | \textit{Diff}+\textit{Emb} & 0.468 & 0.132 & 0.230 & 0.427 & 0.883 \\
\modeltllama3v3chat-70B & $\mathcal{A_{ER}}$ | - | \textit{Diff}+\textit{Emb} & 0.646 & 0.257 & 0.190 & 0.399 & 0.927 \\
\modeltQwen2.5_72B & $\mathcal{A_{ER}}$ | - | \textit{Diff}+\textit{Emb} & 0.546 & 0.246 & 0.210 & 0.416 & 0.903 \\
\bottomrule
\end{tabular}
\end{table}



In [132]:
summary_ood, summary_ood_formatted = summary_table(metrics_ood, distribution='OOD')
summary_ood.round(3)

/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/3628073018.py:66: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["engine", "method_id"])
/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/3628073018.py:74: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("engine", as_index=False)


metric,engine,method_id,original-error,error@MAX-ERG,PVR@MAX-ERG,SAR@MAX-ERG,ERR@MAX-ERG
0,gemini-3-pro-preview,BB | self_consistency | -,0.122,0.021,0.420,0.482,0.855
1,GPT5.2,BB | multi_step | -,0.504,0.219,0.330,0.538,0.881
2,llama3v3chat-70B,BB | self_probing | -,0.694,0.411,0.185,0.417,0.910
3,Qwen2.5_72B,ASAR | CoT | difficulty+demands,0.546,0.343,0.145,0.331,0.913


In [133]:
print(summary_ood_formatted)

\begin{table}
\caption{Summary table of best methods per model based on median out of distribution ERG@MAX-ERG across benchmarks.}
\label{tab:summary_table_best_methods_per_model}
\begin{tabular}{llrrrrr}
\toprule
engine & method_id & \textbf{Orig. Error}$\downarrow$ & \textbf{Err}$\downarrow$ & \textbf{Cov}$\uparrow$ & \textbf{SAR}$\uparrow$ & \textbf{URR}$\uparrow$ \\
\midrule
\modeltgemini-3-pro-preview & Black-Box | \textit{Self-Cons} | - & 0.122 & 0.021 & 0.420 & 0.482 & 0.855 \\
\modeltGPT5.2 & Black-Box | \textit{Multi-Step} | - & 0.504 & 0.219 & 0.330 & 0.538 & 0.881 \\
\modeltllama3v3chat-70B & Black-Box | \textit{Self-Prob} | - & 0.694 & 0.411 & 0.185 & 0.417 & 0.910 \\
\modeltQwen2.5_72B & $\mathcal{A_{AR}}$ | \textit{CoT} | \textit{Diff}+\textit{DeLeAn} & 0.546 & 0.343 & 0.145 & 0.331 & 0.913 \\
\bottomrule
\end{tabular}
\end{table}



#### Table 2 - Median FOC-AUC results across models per benchmark for selected methods and feature sets.

In [146]:
# median across engines and benchmarks per method, variant and feature set


def format_table2(df, distribution='ID', is_selection=False):
    formatted = df.copy().reset_index()
    formatted['method'] = formatted['method'].map(METHOD_FORMATTER)
    formatted['variant'] = formatted['variant'].map(VARIANT_FORMATTER)
    formatted['features'] = formatted['features'].apply(format_feats)
    formatted.set_index(['method', 'variant', 'features'], inplace=True)
    formatted = bold_max_per_method_and_benchmark(formatted.round(3))
    formatted.index.names = [r'$\mathbf{\mathfrak{m}}$', 'Prompt', r'$\phi(\mu)$']
    formatted.index.name = None
    formatted.columns.name = None
    caption = f"%s median FOC-AUC results across models per benchmark."
    if distribution == 'IID':
        caption = caption % "In distribution"
    else:
        caption = caption % "Out of distribution"
    label = f"tab:foc_auc_median_across_benchmarks%s_{distribution.lower()}"
    label = label % ("_selected" if is_selection else "")
    formatted = formatted.to_latex(
        escape=False,
        na_rep="nan",
        caption=caption,
        label=label,
        float_format="%.3f",
        bold_rows=False,
        column_format=('c'  * len(formatted.columns))
    )
    return formatted


def table2_long(metrics, distribution='ID'):
    long = (
        metrics[(metrics['metric'] == 'FOC-AUC')]
            .groupby(['method', 'variant', 'features', 'benchmark'], observed=True)['value'].median()
            .reset_index()
            .pivot(index=['method', 'variant', 'features'], columns='benchmark', values='value')
    )

    long['median'] = long.median(axis=1)
    return long, format_table2(long, distribution=distribution)

def table2_selection(long, distribution='IID'):
    selection = [
        ('ASER', '-', 'embedding+demands'),
        ('ASER', '-', 'difficulty+embedding+demands'),
        ('ASAR', 'CoT', 'difficulty+embedding+demands'),
        ('ASAR', 'self_probing', 'difficulty+embedding+demands'),
        ('BB', 'CoT', '-'),
        ('BB', 'self_consistency', '-'),
        ('GB', 'SAR', '-'),
    ]
    selection = long.loc[selection, :]
    return selection, format_table2(selection, distribution=distribution, is_selection=True)

In [147]:
table2_long_iid, table2_long_iid_formatted = table2_long(metrics_iid, distribution='IID')
table2_selection_iid, table2_selection_iid_formatted = table2_selection(table2_long_iid, distribution='IID')
table2_selection_iid.round(3)

/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.743}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.loc[mask, col] = (
/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.628}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.loc[mask, col] = (
/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.719}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.lo

benchmark                                             addition  anagram  \
method variant          features                                          
ASER   -                embedding+demands                0.733    0.626   
                        difficulty+embedding+demands     0.739    0.628   
ASAR   CoT              difficulty+embedding+demands     0.703    0.620   
       self_probing     difficulty+embedding+demands     0.636    0.612   
BB     CoT              -                                0.508    0.623   
       self_consistency -                                0.649    0.683   
GB     SAR              -                                0.682    0.638   

benchmark                                             locality  science  \
method variant          features                                          
ASER   -                embedding+demands                0.693    0.793   
                        difficulty+embedding+demands     0.714    0.794   
ASAR   CoT              difficulty+embedding+demands     0.655    0.808   
       self_probing     difficulty+embedding+demands     0.729    0.803   
BB     CoT              -                                0.636    0.727   
       self_consistency -                                0.664    0.693   
GB     SAR              -                                0.628    0.697   

benchmark                                             logic_puzzles  geometry  \
method variant          features                                                
ASER   -                embedding+demands                     0.864     0.795   
                        difficulty+embedding+demands          0.870     0.798   
ASAR   CoT              difficulty+embedding+demands          0.780     0.744   
       self_probing     difficulty+embedding+demands          0.782     0.787   
BB     CoT              -                                     0.560     0.711   
       self_consistency -                                     0.535     0.582   
GB     SAR              -                                       NaN     0.595   

benchmark                                             median  
method variant          features                              
ASER   -                embedding+demands              0.763  
                        difficulty+embedding+demands   0.767  
ASAR   CoT              difficulty+embedding+demands   0.724  
       self_probing     difficulty+embedding+demands   0.756  
BB     CoT              -                              0.630  
       self_consistency -                              0.657  
GB     SAR              -                              0.638

In [148]:
print(table2_selection_iid_formatted)

\begin{table}
\caption{In distribution median FOC-AUC results across models per benchmark.}
\label{tab:foc_auc_median_across_benchmarks_selected_iid}
\begin{tabular}{ccccccc}
\toprule
 &  &  & addition & anagram & locality & science & logic_puzzles & geometry & median \\
$\mathbf{\mathfrak{m}}$ & Prompt & $\phi(\mu)$ &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{2}{*}{$\mathcal{A_{ER}}$} & \multirow[t]{2}{*}{nan} & \textit{Emb}+\textit{DeLeAn} & 0.733 & 0.626 & 0.693 & 0.793 & 0.864 & 0.795 & 0.763 \\
 &  & \textit{Diff}+\textit{Emb}+\textit{DeLeAn} & \textbf{0.739} & \textbf{0.628} & \textbf{0.714} & \textbf{0.794} & \textbf{0.870} & \textbf{0.798} & \textbf{0.767} \\
\cline{1-10} \cline{2-10}
\multirow[t]{2}{*}{$\mathcal{A_{AR}}$} & \textit{CoT} & \textit{Diff}+\textit{Emb}+\textit{DeLeAn} & \textbf{0.703} & \textbf{0.620} & 0.655 & \textbf{0.808} & 0.780 & 0.744 & 0.724 \\
\cline{2-10}
 & \textit{Self-Prob} & \textit{Diff}+\textit{Emb}+\textit{DeLeAn} & 0.636 & 0.612 & \textbf{0.729

In [150]:
table2_long_iid.round(3)

benchmark                                             addition  anagram  \
method variant          features                                          
ASER   -                difficulty                       0.684    0.595   
                        embedding                        0.739    0.610   
                        demands                          0.582    0.603   
                        difficulty+embedding             0.743    0.615   
                        difficulty+demands               0.664    0.586   
                        embedding+demands                0.733    0.626   
                        difficulty+embedding+demands     0.739    0.628   
ASAR   vanilla          difficulty                       0.621    0.695   
                        embedding                        0.684    0.704   
                        demands                          0.561    0.585   
                        difficulty+embedding             0.690    0.703   
                        difficulty+demands               0.651    0.583   
                        embedding+demands                0.678    0.627   
                        difficulty+embedding+demands     0.696    0.628   
       CoT              difficulty                       0.645    0.700   
                        embedding                        0.687    0.703   
                        demands                          0.571    0.595   
                        difficulty+embedding             0.699    0.703   
                        difficulty+demands               0.634    0.589   
                        embedding+demands                0.694    0.624   
                        difficulty+embedding+demands     0.703    0.620   
       self_probing     difficulty                       0.650    0.701   
                        embedding                        0.627    0.695   
                        demands                          0.552    0.590   
                        difficulty+embedding             0.630    0.692   
                        difficulty+demands               0.620    0.593   
                        embedding+demands                0.626    0.617   
                        difficulty+embedding+demands     0.636    0.612   
       multi_step       difficulty                       0.620    0.701   
                        embedding                        0.693    0.705   
                        demands                          0.561    0.604   
                        difficulty+embedding             0.702    0.709   
                        difficulty+demands               0.636    0.600   
                        embedding+demands                0.691    0.619   
                        difficulty+embedding+demands     0.701    0.619   
       top_k            difficulty                       0.602    0.666   
                        embedding                        0.667    0.655   
                        demands                          0.558    0.594   
                        difficulty+embedding             0.671    0.659   
                        difficulty+demands               0.643    0.584   
                        embedding+demands                0.664    0.584   
                        difficulty+embedding+demands     0.681    0.587   
BB     vanilla          -                                0.467    0.722   
       CoT              -                                0.508    0.623   
       self_probing     -                                0.256    0.353   
       multi_step       -                                0.584    0.618   
       top_k            -                                0.411    0.562   
       self_consistency -                                0.649    0.683   
GB     SAR              -                                0.682    0.638   
       semantic_energy  -                                0.502    0.419   

benchmark                                             locality  science  \
method variant          

In [151]:
print(table2_long_iid_formatted)

\begin{table}
\caption{In distribution median FOC-AUC results across models per benchmark.}
\label{tab:foc_auc_median_across_benchmarks_iid}
\begin{tabular}{ccccccc}
\toprule
 &  &  & addition & anagram & locality & science & logic_puzzles & geometry & median \\
$\mathbf{\mathfrak{m}}$ & Prompt & $\phi(\mu)$ &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{7}{*}{$\mathcal{A_{ER}}$} & \multirow[t]{7}{*}{nan} & \textit{Diff} & 0.684 & 0.595 & 0.590 & 0.781 & 0.548 & 0.515 & 0.593 \\
 &  & \textit{Emb} & 0.739 & 0.610 & 0.701 & 0.797 & 0.869 & \textbf{0.798} & 0.768 \\
 &  & \textit{DeLeAn} & 0.582 & 0.603 & 0.512 & 0.756 & 0.760 & 0.691 & 0.647 \\
 &  & \textit{Diff}+\textit{Emb} & \textbf{0.743} & 0.615 & \textbf{0.719} & \textbf{0.801} & \textbf{0.874} & \textbf{0.798} & \textbf{0.771} \\
 &  & \textit{Diff}+\textit{DeLeAn} & 0.664 & 0.586 & 0.590 & 0.769 & 0.773 & 0.699 & 0.681 \\
 &  & \textit{Emb}+\textit{DeLeAn} & 0.733 & 0.626 & 0.693 & 0.793 & 0.864 & 0.795 & 0.763 \\
 &  & \textit{

In [154]:
table2_long_ood, table2_long_ood_formatted = table2_long(metrics_ood, distribution='OOD')
table2_selection_ood, table2_selection_ood_formatted = table2_selection(table2_long_ood, distribution='OOD')
table2_selection_ood.round(3)

/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.567}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.loc[mask, col] = (
/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.599}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.loc[mask, col] = (
/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.511}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.lo

benchmark                                             addition  anagram  \
method variant          features                                          
ASER   -                embedding+demands                0.520    0.598   
                        difficulty+embedding+demands     0.533    0.599   
ASAR   CoT              difficulty+embedding+demands     0.616    0.615   
       self_probing     difficulty+embedding+demands     0.607    0.609   
BB     CoT              -                                0.508    0.623   
       self_consistency -                                0.649    0.683   
GB     SAR              -                                0.682    0.638   

benchmark                                             locality  science  \
method variant          features                                          
ASER   -                embedding+demands                0.498    0.609   
                        difficulty+embedding+demands     0.483    0.600   
ASAR   CoT              difficulty+embedding+demands     0.514    0.719   
       self_probing     difficulty+embedding+demands     0.502    0.745   
BB     CoT              -                                0.636    0.727   
       self_consistency -                                0.664    0.693   
GB     SAR              -                                0.628    0.697   

benchmark                                             logic_puzzles  geometry  \
method variant          features                                                
ASER   -                embedding+demands                     0.590     0.620   
                        difficulty+embedding+demands          0.628     0.590   
ASAR   CoT              difficulty+embedding+demands          0.649     0.609   
       self_probing     difficulty+embedding+demands          0.631     0.610   
BB     CoT              -                                     0.560     0.711   
       self_consistency -                                     0.535     0.582   
GB     SAR              -                                       NaN     0.595   

benchmark                                             median  
method variant          features                              
ASER   -                embedding+demands              0.594  
                        difficulty+embedding+demands   0.594  
ASAR   CoT              difficulty+embedding+demands   0.615  
       self_probing     difficulty+embedding+demands   0.609  
BB     CoT              -                              0.630  
       self_consistency -                              0.657  
GB     SAR              -                              0.638

In [155]:
print(table2_selection_iid_formatted)

\begin{table}
\caption{In distribution median FOC-AUC results across models per benchmark.}
\label{tab:foc_auc_median_across_benchmarks_selected_iid}
\begin{tabular}{ccccccc}
\toprule
 &  &  & addition & anagram & locality & science & logic_puzzles & geometry & median \\
$\mathbf{\mathfrak{m}}$ & Prompt & $\phi(\mu)$ &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{2}{*}{$\mathcal{A_{ER}}$} & \multirow[t]{2}{*}{nan} & \textit{Emb}+\textit{DeLeAn} & 0.733 & 0.626 & 0.693 & 0.793 & 0.864 & 0.795 & 0.763 \\
 &  & \textit{Diff}+\textit{Emb}+\textit{DeLeAn} & \textbf{0.739} & \textbf{0.628} & \textbf{0.714} & \textbf{0.794} & \textbf{0.870} & \textbf{0.798} & \textbf{0.767} \\
\cline{1-10} \cline{2-10}
\multirow[t]{2}{*}{$\mathcal{A_{AR}}$} & \textit{CoT} & \textit{Diff}+\textit{Emb}+\textit{DeLeAn} & \textbf{0.703} & \textbf{0.620} & 0.655 & \textbf{0.808} & 0.780 & 0.744 & 0.724 \\
\cline{2-10}
 & \textit{Self-Prob} & \textit{Diff}+\textit{Emb}+\textit{DeLeAn} & 0.636 & 0.612 & \textbf{0.729

In [156]:
table2_long_ood.round(3)

benchmark                                             addition  anagram  \
method variant          features                                          
ASER   -                difficulty                       0.456    0.381   
                        embedding                        0.508    0.536   
                        demands                          0.506    0.565   
                        difficulty+embedding             0.567    0.485   
                        difficulty+demands               0.440    0.562   
                        embedding+demands                0.520    0.598   
                        difficulty+embedding+demands     0.533    0.599   
ASAR   vanilla          difficulty                       0.612    0.601   
                        embedding                        0.528    0.535   
                        demands                          0.472    0.581   
                        difficulty+embedding             0.592    0.580   
                        difficulty+demands               0.548    0.613   
                        embedding+demands                0.524    0.590   
                        difficulty+embedding+demands     0.586    0.603   
       CoT              difficulty                       0.626    0.599   
                        embedding                        0.543    0.600   
                        demands                          0.499    0.558   
                        difficulty+embedding             0.632    0.673   
                        difficulty+demands               0.501    0.588   
                        embedding+demands                0.531    0.582   
                        difficulty+embedding+demands     0.616    0.615   
       self_probing     difficulty                       0.609    0.594   
                        embedding                        0.510    0.564   
                        demands                          0.510    0.558   
                        difficulty+embedding             0.577    0.683   
                        difficulty+demands               0.483    0.600   
                        embedding+demands                0.537    0.598   
                        difficulty+embedding+demands     0.607    0.609   
       multi_step       difficulty                       0.601    0.602   
                        embedding                        0.526    0.557   
                        demands                          0.479    0.547   
                        difficulty+embedding             0.559    0.599   
                        difficulty+demands               0.459    0.593   
                        embedding+demands                0.539    0.601   
                        difficulty+embedding+demands     0.586    0.612   
       top_k            difficulty                       0.601    0.604   
                        embedding                        0.495    0.541   
                        demands                          0.506    0.589   
                        difficulty+embedding             0.554    0.588   
                        difficulty+demands               0.521    0.583   
                        embedding+demands                0.516    0.587   
                        difficulty+embedding+demands     0.513    0.626   
BB     vanilla          -                                0.467    0.722   
       CoT              -                                0.508    0.623   
       self_probing     -                                0.256    0.353   
       multi_step       -                                0.584    0.618   
       top_k            -                                0.411    0.562   
       self_consistency -                                0.649    0.683   
GB     SAR              -                                0.682    0.638   
       semantic_energy  -                                0.502    0.419   

benchmark                                             locality  science  \
method variant          

In [158]:
print(table2_long_ood_formatted)

\begin{table}
\caption{Out of distribution median FOC-AUC results across models per benchmark.}
\label{tab:foc_auc_median_across_benchmarks_ood}
\begin{tabular}{ccccccc}
\toprule
 &  &  & addition & anagram & locality & science & logic_puzzles & geometry & median \\
$\mathbf{\mathfrak{m}}$ & Prompt & $\phi(\mu)$ &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{7}{*}{$\mathcal{A_{ER}}$} & \multirow[t]{7}{*}{nan} & \textit{Diff} & 0.456 & 0.381 & \textbf{0.511} & 0.484 & 0.536 & 0.528 & 0.497 \\
 &  & \textit{Emb} & 0.508 & 0.536 & 0.503 & 0.583 & 0.584 & 0.613 & 0.560 \\
 &  & \textit{DeLeAn} & 0.506 & 0.565 & 0.508 & 0.548 & 0.524 & 0.517 & 0.521 \\
 &  & \textit{Diff}+\textit{Emb} & \textbf{0.567} & 0.485 & 0.490 & 0.547 & 0.610 & 0.548 & 0.548 \\
 &  & \textit{Diff}+\textit{DeLeAn} & 0.440 & 0.562 & 0.504 & \textbf{0.647} & 0.533 & 0.528 & 0.531 \\
 &  & \textit{Emb}+\textit{DeLeAn} & 0.520 & 0.598 & 0.498 & 0.609 & 0.590 & \textbf{0.620} & \textbf{0.594} \\
 &  & \textit{Diff}+\textit{

#### Supplemenetary Tables A - FOC-AUC results per engine across tasks, prompts and asessor feature sets.

In [163]:
def suptableA(metrics, engine): 
    table = (
        metrics[metrics['metric'] == 'FOC-AUC']
            .pivot(index=['engine', 'method', 'variant', 'features'], columns='benchmark', values='value')
            .xs(engine, level='engine')
    )
    return table

def format_suptableA(table, engine, distribution='IID'):
    formatted = table.copy().reset_index()
    formatted['method'] = formatted['method'].map(METHOD_FORMATTER)
    formatted['variant'] = formatted['variant'].map(VARIANT_FORMATTER)
    formatted['features'] = formatted['features'].apply(format_feats)
    formatted.set_index(['method', 'variant', 'features'], inplace=True)
    formatted = bold_max_per_method_and_benchmark(formatted.round(3))
    formatted.index.names = [r'$\mathbf{\mathfrak{m}}$', 'Prompt', r'$\phi(\mu)$']
    formatted.index.name = None
    formatted.columns.name = None
    caption = (
    r"%s FOC-AUC results for \modelt{%s} across tasks, prompts and assessor feature sets. "
    "Rows correspond to assessor configurations and prompting strategies, columns to benchmark tasks."
    )
    if distribution == 'IID':
        caption = caption % ("In distribution", engine)
    else:
        caption = caption % ("Out of distribution", engine)
    label = "tab:foc_auc_id_%s"
    formatted = formatted.to_latex(
        escape=False,
        na_rep="nan",
        caption=caption,
        label=label,
        float_format="%.3f",
        bold_rows=False,
        column_format=('c'  * len(formatted.columns))
    )
    return formatted

In [164]:
distribution = 'IID'
for engine in ENGINES:
    print(f"%% [{engine.upper()}]\n")
    table = suptableA(metrics_iid, engine)
    formatted = format_suptableA(table, engine, distribution=distribution)
    print(formatted)
    

%% [GEMINI-3-PRO-PREVIEW]

\begin{table}
\caption{In distribution FOC-AUC results for \modelt{gemini-3-pro-preview} across tasks, prompts and assessor feature sets. Rows correspond to assessor configurations and prompting strategies, columns to benchmark tasks.}
\label{tab:foc_auc_id_%s}
\begin{tabular}{cccccc}
\toprule
 &  &  & addition & anagram & locality & science & logic_puzzles & geometry \\
$\mathbf{\mathfrak{m}}$ & Prompt & $\phi(\mu)$ &  &  &  &  &  &  \\
\midrule
\multirow[t]{7}{*}{$\mathcal{A_{ER}}$} & \multirow[t]{7}{*}{nan} & \textit{Diff} & \textbf{0.694} & \textbf{0.180} & 0.594 & 0.671 & 0.566 & 0.630 \\
 &  & \textit{Emb} & 0.675 & \textbf{0.180} & 0.701 & 0.698 & 0.937 & \textbf{0.844} \\
 &  & \textit{DeLeAn} & 0.566 & nan & 0.511 & 0.699 & 0.759 & 0.683 \\
 &  & \textit{Diff}+\textit{Emb} & 0.689 & \textbf{0.180} & \textbf{0.720} & 0.705 & \textbf{0.942} & 0.842 \\
 &  & \textit{Diff}+\textit{DeLeAn} & 0.658 & nan & 0.588 & 0.683 & 0.782 & 0.717 \\
 &  & \textit{Emb

/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.694}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.loc[mask, col] = (
/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.180}' '\\textbf{0.180}' '\\textbf{0.180}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.loc[mask, col] = (
/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.720}']' has dtype incompatible with float64, please explicitly cast to a

In [165]:
distribution = 'OOD'
for engine in ENGINES:
    print(f"%% [{engine.upper()}]\n")
    table = suptableA(metrics_ood, engine)
    formatted = format_suptableA(table, engine, distribution=distribution)
    print(formatted)
    

%% [GEMINI-3-PRO-PREVIEW]

\begin{table}
\caption{Out of distribution FOC-AUC results for \modelt{gemini-3-pro-preview} across tasks, prompts and assessor feature sets. Rows correspond to assessor configurations and prompting strategies, columns to benchmark tasks.}
\label{tab:foc_auc_id_%s}
\begin{tabular}{cccccc}
\toprule
 &  &  & addition & anagram & locality & science & logic_puzzles & geometry \\
$\mathbf{\mathfrak{m}}$ & Prompt & $\phi(\mu)$ &  &  &  &  &  &  \\
\midrule
\multirow[t]{7}{*}{$\mathcal{A_{ER}}$} & \multirow[t]{7}{*}{nan} & \textit{Diff} & 0.347 & \textbf{0.250} & 0.516 & 0.485 & 0.607 & \textbf{0.634} \\
 &  & \textit{Emb} & 0.500 & 0.235 & \textbf{0.517} & 0.609 & 0.564 & 0.461 \\
 &  & \textit{DeLeAn} & 0.466 & nan & 0.484 & 0.715 & 0.495 & 0.489 \\
 &  & \textit{Diff}+\textit{Emb} & \textbf{0.544} & 0.180 & 0.499 & 0.553 & 0.617 & 0.518 \\
 &  & \textit{Diff}+\textit{DeLeAn} & 0.401 & nan & 0.493 & \textbf{0.737} & 0.502 & 0.553 \\
 &  & \textit{Emb}+\textit{DeLe

/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.544}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.loc[mask, col] = (
/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.250}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.loc[mask, col] = (
/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.517}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.lo

#### Supplementary Tables B - summary of rejection metrics at ORP for `engine` on `benchmark`

In [172]:
def summary_long(df):
    # Métricas que quieres en la tabla final
    target_metrics = [
        "original-error",
        "ACC@MAX-ERG",
        "PVR@MAX-ERG",
        "SAR@MAX-ERG",
        "ERR@MAX-ERG",
    ]
    df_filt = df[df["metric"].isin(target_metrics)].copy()
    table = (
        df_filt
        .pivot_table(
            index=["engine", "benchmark", "method", "variant", "features"],
            columns="metric",
            values="value",
            aggfunc="mean"  
        ) 
        .reset_index()
    )
    table = table[
        [
            "engine",
            "benchmark",
            "method",
            "variant",
            "features",
            "original-error",
            "ACC@MAX-ERG",
            "PVR@MAX-ERG",
            "SAR@MAX-ERG",
            "ERR@MAX-ERG",
        ]
    ]
    table['error@MAX-ERG'] = 1 - table['ACC@MAX-ERG']
    return table

def format_suptableB_long(summary_long, distribution='ID'):
    formatted = summary_long.copy()
    formatted['method'] = formatted['method'].map(METHOD_FORMATTER)
    formatted['variant'] = formatted['variant'].map(VARIANT_FORMATTER)
    formatted['features'] = formatted['features'].apply(format_feats)
    formatted.set_index(['method', 'variant', 'features'], inplace=True)
    formatted = bold_max_per_method_and_benchmark(
        formatted.round(3),
        subset=['original-error', 'error@MAX-ERG', 'PVR@MAX-ERG', 'SAR@MAX-ERG', 'ERR@MAX-ERG'],
        direction={
            'original-error': 'min',
            'error@MAX-ERG': 'min',
            'PVR@MAX-ERG': 'max',
            'SAR@MAX-ERG': 'max',
            'ERR@MAX-ERG': 'max'
        }
    )
    formatted.index.names = [r'$\mathbf{\mathfrak{m}}$', 'Prompt', r'$\phi(\mu)$']
    formatted.index.name = None
    formatted.columns.name = None
    caption = (
    r"%s rejection metrics at optimal threshold for \modelt{%s} on \modelt{%s} "
    )
    label = "tab:summary_rejection_%s_%s_iid"
    if distribution == 'IID':
        caption = caption % ("In distribution", engine, benchmark)
    else:
        caption = caption % ("Out of distribution", engine, benchmark)
    label = f"tab:foc_auc_all_methods_long_{distribution.lower()}"
    formatted = formatted.to_latex(
        escape=False,
        na_rep="nan",
        caption=caption,
        label=label,
        float_format="%.3f",
        bold_rows=False,
        column_format=('c'  * len(formatted.columns))
    )
    return formatted

In [173]:
summary_iid = summary_long(metrics_iid)
for engine in ENGINES:
    for benchmark in BENCHMARKS:
        print(f"%% [{engine.upper()} - {benchmark}]\n")
        subset = summary_iid[
            (summary_iid['engine'] == engine) &
            (summary_iid['benchmark'] == benchmark)
        ]
        print(format_suptableB_long(subset))

%% [GEMINI-3-PRO-PREVIEW - addition]

\begin{table}
\caption{Out of distribution rejection metrics at optimal threshold for \modelt{gemini-3-pro-preview} on \modelt{addition} }
\label{tab:foc_auc_all_methods_long_id}
\begin{tabular}{cccccccc}
\toprule
 &  &  & engine & benchmark & original-error & ACC@MAX-ERG & PVR@MAX-ERG & SAR@MAX-ERG & ERR@MAX-ERG & error@MAX-ERG \\
$\mathbf{\mathfrak{m}}$ & Prompt & $\phi(\mu)$ &  &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{7}{*}{$\mathcal{A_{ER}}$} & \multirow[t]{7}{*}{nan} & \textit{Diff} & gemini-3-pro-preview & addition & \textbf{0.061} & 0.995 & 0.270 & 0.286 & \textbf{0.979} & \textbf{0.005} \\
 &  & \textit{Emb} & gemini-3-pro-preview & addition & \textbf{0.061} & 0.977 & \textbf{0.420} & \textbf{0.437} & 0.839 & 0.023 \\
 &  & \textit{DeLeAn} & gemini-3-pro-preview & addition & 0.062 & 0.956 & 0.380 & 0.387 & 0.729 & 0.044 \\
 &  & \textit{Diff}+\textit{Emb} & gemini-3-pro-preview & addition & \textbf{0.061} & 0.985 & 0.330 & 0.346 & 0.91

/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.061}' '\\textbf{0.061}' '\\textbf{0.061}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.loc[mask, col] = (
/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.420}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.loc[mask, col] = (
/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.437}']' has dtype incompatible with float64, please explicitly cast to a

\begin{table}
\caption{Out of distribution rejection metrics at optimal threshold for \modelt{llama3v3chat-70B} on \modelt{geometry} }
\label{tab:foc_auc_all_methods_long_id}
\begin{tabular}{cccccccc}
\toprule
 &  &  & engine & benchmark & original-error & ACC@MAX-ERG & PVR@MAX-ERG & SAR@MAX-ERG & ERR@MAX-ERG & error@MAX-ERG \\
$\mathbf{\mathfrak{m}}$ & Prompt & $\phi(\mu)$ &  &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{7}{*}{$\mathcal{A_{ER}}$} & \multirow[t]{7}{*}{nan} & \textit{Diff} & llama3v3chat-70B & geometry & 0.732 & 0.417 & 0.040 & 0.062 & \textbf{0.968} & 0.583 \\
 &  & \textit{Emb} & llama3v3chat-70B & geometry & 0.731 & 0.696 & 0.180 & 0.467 & 0.925 & 0.304 \\
 &  & \textit{DeLeAn} & llama3v3chat-70B & geometry & \textbf{0.729} & 0.605 & \textbf{0.200} & 0.447 & 0.892 & 0.395 \\
 &  & \textit{Diff}+\textit{Emb} & llama3v3chat-70B & geometry & 0.731 & 0.729 & 0.160 & 0.434 & 0.941 & \textbf{0.271} \\
 &  & \textit{Diff}+\textit{DeLeAn} & llama3v3chat-70B & geometry & \tex

/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.454}' '\\textbf{0.454}' '\\textbf{0.454}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.loc[mask, col] = (
/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.410}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.loc[mask, col] = (
/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.521}']' has dtype incompatible with float64, please explicitly cast to a

In [175]:
summary_ood = summary_long(metrics_ood)
for engine in ENGINES:
    for benchmark in BENCHMARKS:
        print(f"%% [{engine.upper()} - {benchmark}]\n")
        subset = summary_ood[
            (summary_ood['engine'] == engine) &
            (summary_ood['benchmark'] == benchmark)
        ]
        print(format_suptableB_long(subset, distribution='OOD'))

%% [GEMINI-3-PRO-PREVIEW - addition]

\begin{table}
\caption{Out of distribution rejection metrics at optimal threshold for \modelt{gemini-3-pro-preview} on \modelt{addition} }
\label{tab:foc_auc_all_methods_long_ood}
\begin{tabular}{cccccccc}
\toprule
 &  &  & engine & benchmark & original-error & ACC@MAX-ERG & PVR@MAX-ERG & SAR@MAX-ERG & ERR@MAX-ERG & error@MAX-ERG \\
$\mathbf{\mathfrak{m}}$ & Prompt & $\phi(\mu)$ &  &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{7}{*}{$\mathcal{A_{ER}}$} & \multirow[t]{7}{*}{nan} & \textit{Diff} & gemini-3-pro-preview & addition & \textbf{0.061} & 0.959 & 0.100 & 0.102 & 0.932 & 0.041 \\
 &  & \textit{Emb} & gemini-3-pro-preview & addition & \textbf{0.061} & 0.968 & 0.090 & 0.093 & 0.953 & 0.032 \\
 &  & \textit{DeLeAn} & gemini-3-pro-preview & addition & 0.062 & 0.947 & 0.030 & 0.031 & 0.974 & 0.053 \\
 &  & \textit{Diff}+\textit{Emb} & gemini-3-pro-preview & addition & \textbf{0.061} & 0.953 & \textbf{0.410} & \textbf{0.416} & 0.682 & 0.047 \\
 &  

/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.061}' '\\textbf{0.061}' '\\textbf{0.061}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.loc[mask, col] = (
/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.410}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.loc[mask, col] = (
/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.416}']' has dtype incompatible with float64, please explicitly cast to a

\begin{table}
\caption{Out of distribution rejection metrics at optimal threshold for \modelt{llama3v3chat-70B} on \modelt{locality} }
\label{tab:foc_auc_all_methods_long_ood}
\begin{tabular}{cccccccc}
\toprule
 &  &  & engine & benchmark & original-error & ACC@MAX-ERG & PVR@MAX-ERG & SAR@MAX-ERG & ERR@MAX-ERG & error@MAX-ERG \\
$\mathbf{\mathfrak{m}}$ & Prompt & $\phi(\mu)$ &  &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{7}{*}{$\mathcal{A_{ER}}$} & \multirow[t]{7}{*}{nan} & \textit{Diff} & llama3v3chat-70B & locality & \textbf{0.561} & 0.467 & 0.290 & 0.309 & 0.724 & 0.533 \\
 &  & \textit{Emb} & llama3v3chat-70B & locality & \textbf{0.561} & 0.457 & 0.400 & 0.417 & 0.613 & 0.543 \\
 &  & \textit{DeLeAn} & llama3v3chat-70B & locality & \textbf{0.561} & 0.449 & 0.670 & 0.686 & 0.343 & 0.551 \\
 &  & \textit{Diff}+\textit{Emb} & llama3v3chat-70B & locality & \textbf{0.561} & 0.442 & \textbf{0.840} & \textbf{0.846} & 0.165 & 0.558 \\
 &  & \textit{Diff}+\textit{DeLeAn} & llama3v3chat-70

/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.186}' '\\textbf{0.186}' '\\textbf{0.186}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.loc[mask, col] = (
/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.490}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_out.loc[mask, col] = (
/var/folders/w6/fm29qcqj6xgc24t18_ngmnd00000gn/T/ipykernel_67117/1053369706.py:58: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['\\textbf{0.550}']' has dtype incompatible with float64, please explicitly cast to a